## `06_hypotheses_testing.ipynb`

This notebook is the **hypothesis-testing step of the pipeline**. It fits the core
**cross-classified mixed-effects models** used in the Results section, with ratings nested
in both **coders** (random intercept) and **tweets** (variance component). It estimates the
stepwise model set (Models 1–6) for each outcome and exports publication-ready outputs.

In addition, it generates the **congruence-effects appendix figures** (two-panel plots)
that visualize (A) predicted ratings by **congruence × source visibility** and (B) **simple
congruence slopes by instruction condition**, based on the same cross-classified modeling
framework.

### Pipeline
`00_get-Xaccounts-MPs.ipynb` → `01_get_twitter_data.ipynb` → `02_prepare_data_for_analysis.ipynb` → `03_irr_and_descriptives.ipynb` → `04_main_models.ipynb` → `06_hypotheses_testing.ipynb`

### What it does
- Loads the final analysis dataset from the Research Drive (`final_merged_dataset_for_analysis.parquet`).
- Ensures **factor ordering** for `instruction_type` and **grand-mean centering** of continuous covariates used in interactions.
- Fits **stepwise cross-classified MixedLMs** for each outcome (`sentiment`, `misinformation`, `toxicity`), including:
  - Model 4: **Congruence × Implicit Bias**
  - Model 5: **Source × Congruence**
  - Model 6: **Instruction × Congruence**
- Uses **patsy-based listwise deletion** per model formula (robust to interactions/categorical coding).
- Extracts and reports **variance components**, **ICCs**, log-likelihood, and fit diagnostics.
- Produces **appendix congruence figures** with fixed-effect predictions and 95% CIs.

### Output
- `output/tables/<outcome>_models_xclass.tex` — LaTeX tables for Models 1–6 (cross-classified MixedLM).
- `output/tables/<outcome>_models_xclass_tidy.csv` — tidy coefficient tables (for audit/reuse).
- `output/figures/<outcome>_appendix_congruence_panels.png` — two-panel appendix figures (source visibility + instruction slopes).


In [1]:
# ============================================================
# Cross-classified mixed models (coder RI + tweet VC)
# - Model 4 includes Congruence × Implicit Bias
# - PolOrient_1_c listed under CONTROL VARIABLES in the LaTeX
# - Centered continuous covariates
# - Robust variance components + ICCs
# - LaTeX + tidy CSV to Research Drive
# ============================================================
import io
import requests
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import patsy  # <- NEW: robust listwise deletion based on the actual model matrix

from rd_utils import webdav_mkdirs, webdav_upload_bytes
import config  # expects BASE_URL, USER, APP_PASSWORD

# -----------------------------
# 1) Load data
# -----------------------------
BASE = f"{config.BASE_URL}/{config.USER}"
DATA_PATH = ("ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/"
             "final_merged_dataset_for_analysis.parquet")
url = f"{BASE}/{DATA_PATH}"

resp = requests.get(url, auth=(config.USER, config.APP_PASSWORD), stream=True)
resp.raise_for_status()
df = pd.read_parquet(io.BytesIO(resp.content))

# -----------------------------
# 2) Prep: factor ordering + centering
# -----------------------------
df['instruction_type'] = pd.Categorical(
    df['instruction_type'],
    categories=['no instructions', 'general instructions', 'tailored instructions'],
    ordered=True
)

if 'PolOrient_1' not in df.columns:
    raise KeyError("PolOrient_1 (0–10 left–right self-identification) not found in dataset.")
df['PolOrient_1'] = pd.to_numeric(df['PolOrient_1'], errors='coerce')

def center_if_present(d, col):
    if col in d.columns:
        d[col + "_c"] = d[col] - d[col].mean(skipna=True)
    return d

for col in [
    'Age', 'PolOrient_1', 'vote_likelihood_score', 'dsc',
    'ImportIssue_1_numeric', 'KnowIssue_1_numeric', 'AttitudeExtr2_combined'
]:
    df = center_if_present(df, col)

# -----------------------------
# 3) Cross-classified fitting utilities
# -----------------------------
def _listwise_drop_patsy(formula, data):
    """
    Robust per-model listwise deletion:
    patsy determines which rows are valid given the full formula
    (handles interactions, categorical terms, transforms, etc.).
    """
    y, X = patsy.dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    return data.loc[X.index].copy()

def _fit_xclass(formula, data, group_col="uid", tweet_col="unit_id",
                reml=True, method="lbfgs", debug=False):

    # Robust listwise deletion based on actual model matrix
    d = _listwise_drop_patsy(formula, data)

    # Guardrails (helpful for weird missingness edge-cases)
    if group_col not in d.columns:
        raise KeyError(f"{group_col} not found after listwise deletion.")
    if tweet_col not in d.columns:
        raise KeyError(f"{tweet_col} not found after listwise deletion.")

    # Fit ladder (REML lbfgs -> ML lbfgs -> ML Nelder-Mead)
    fit_meta = {"attempt": None, "reml": None, "method": None}
    try:
        fit_meta.update({"attempt": 1, "reml": reml, "method": method})
        m = smf.mixedlm(
            formula, data=d,
            groups=d[group_col],              # coder random intercept
            re_formula="1",
            vc_formula={"tweet": f"0 + C({tweet_col})"}  # tweet variance component
        ).fit(reml=reml, method=method)
    except Exception:
        try:
            fit_meta.update({"attempt": 2, "reml": False, "method": method})
            m = smf.mixedlm(
                formula, data=d,
                groups=d[group_col],
                re_formula="1",
                vc_formula={"tweet": f"0 + C({tweet_col})"}
            ).fit(reml=False, method=method)
        except Exception:
            fit_meta.update({"attempt": 3, "reml": False, "method": "nm"})
            m = smf.mixedlm(
                formula, data=d,
                groups=d[group_col],
                re_formula="1",
                vc_formula={"tweet": f"0 + C({tweet_col})"}
            ).fit(reml=False, method="nm", maxiter=2000, disp=False)

    # Attach fit metadata (doesn't change results; helps reporting/debugging)
    m._fit_meta = {
        **fit_meta,
        "n_obs": int(getattr(m, "nobs", len(d))),
        "n_coders": int(d[group_col].nunique()),
        "n_tweets": int(d[tweet_col].nunique()),
    }

    if debug:
        vcomp = getattr(m, "vcomp", None)
        print("[DEBUG] attempt=", m._fit_meta.get("attempt"),
              "reml=", m._fit_meta.get("reml"),
              "method=", m._fit_meta.get("method"),
              "n=", m._fit_meta.get("n_obs"))
        print("[DEBUG] cov_re shape=", np.asarray(getattr(m, 'cov_re', np.empty((0,0)))).shape,
              "vcomp=", vcomp, "scale=", getattr(m,'scale',np.nan))

    return m

def fit_stepwise_models_xclass(data, variable_name, debug=False):
    d = data[data['variable'] == variable_name].copy()

    formulas = [
        # M1: baseline (no dsc, no congruence)
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + instruction_type + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",

        # M2: + Implicit Bias
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + instruction_type + dsc_c + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",

        # M3: + Ideological Congruence
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + instruction_type + vote_likelihood_score_c + dsc_c + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",

        # M4: fully adjusted + Congruence × Bias (requested)
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + vote_likelihood_score_c + instruction_type + dsc_c + "
        "vote_likelihood_score_c:dsc_c + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",

        # M5: source × congruence
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown * vote_likelihood_score_c + instruction_type + dsc_c + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",

        # M6: instruction × congruence
        "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + vote_likelihood_score_c * instruction_type + dsc_c + "
        "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
        "Edu_binary + Werk_binair + Etniciteit_binair",
    ]

    models = []
    for f in formulas:
        m = _fit_xclass(f, d, group_col="uid", tweet_col="unit_id",
                        reml=True, method="lbfgs", debug=debug)
        models.append(m)
    return models

# -----------------------------
# 4) Names + variance/ICC extractors
# -----------------------------
def rename_predictors():
    return {
        'Intercept': 'Intercept',
        # Treatments
        'instruction_type[T.general instructions]': 'General Instructions',
        'instruction_type[T.tailored instructions]': 'Tailored Instructions',
        'source_shown': 'Source Shown',
        # Interactions
        'source_shown:vote_likelihood_score_c': 'Source × Congruence',
        'vote_likelihood_score_c:instruction_type[T.general instructions]': 'Congruence × General Instr.',
        'vote_likelihood_score_c:instruction_type[T.tailored instructions]':  'Congruence × Tailored Instr.',
        'vote_likelihood_score_c:dsc_c': 'Congruence × Bias',
        # Focal covariates (centered)
        'dsc_c': 'Implicit Bias (centered)',
        'vote_likelihood_score_c': 'Ideological Congruence (centered)',
        'PolOrient_1_c': 'Left–Right Self-identification (centered)',
        # Controls
        'Male': 'Male',
        'Age_c': 'Age (centered)',
        'ImportIssue_1_numeric_c': 'Issue Importance (centered)',
        'KnowIssue_1_numeric_c': 'Issue Knowledge (centered)',
        'AttitudeExtr2_combined_c': 'Anti-immigrant Attitude (centered)',
        'Edu_binary': 'Higher Educated',
        'Werk_binair': 'Employment (1 = employed)',
        'Etniciteit_binair': 'Ethnicity (1 = Dutch)',
    }

def sig_star(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def _var_coder(m):
    try:
        arr = np.asarray(getattr(m, "cov_re", None), dtype=float)
        if arr.size:
            return float(arr.ravel()[0])
    except Exception:
        pass
    return np.nan

def _var_tweet(m):
    vc = getattr(m, "vcomp", None)
    if vc is None:
        return np.nan
    arr = np.asarray(vc, dtype=float).ravel()
    if arr.size == 1:
        return float(arr[0])
    names = getattr(m.model, "vcomp_names", None)
    if names is not None:
        for i, nm in enumerate(names):
            if "tweet" in str(nm).lower() or "unit" in str(nm).lower():
                return float(arr[i])
    # fallback: if naming fails, return first component
    return float(arr[0]) if arr.size else np.nan

def _resid_var(m):
    try:
        return float(m.scale)
    except Exception:
        return np.nan

def _icc_coder(m):
    vc, vt, ve = _var_coder(m), _var_tweet(m), _resid_var(m)
    den = vc + vt + ve
    return float(vc/den) if den > 0 else np.nan

def _icc_tweet(m):
    vc, vt, ve = _var_coder(m), _var_tweet(m), _resid_var(m)
    den = vc + vt + ve
    return float(vt/den) if den > 0 else np.nan

# -----------------------------
# 5) LaTeX generator (PolOrient_1_c under CONTROL VARIABLES)
# -----------------------------
def generate_latex_table_xclass(models, rename_map, variable_name):
    sections = {
        'Main Effects': [
            'instruction_type[T.general instructions]',
            'instruction_type[T.tailored instructions]',
            'source_shown',
            'dsc_c',
        ],
        'H1: Effects of Ideological Congruence': ['vote_likelihood_score_c'],
        'H2: Moderating Effects of Implicit Bias': ['vote_likelihood_score_c:dsc_c'],
        'H3: Moderating Effects of Source Visibility': ['source_shown:vote_likelihood_score_c'],
        'H4: Moderating Effects of Instruction Type': [
            'vote_likelihood_score_c:instruction_type[T.general instructions]',
            'vote_likelihood_score_c:instruction_type[T.tailored instructions]'
        ],
        'Control Variables': [
            'Male', 'Age_c', 'PolOrient_1_c',
            'ImportIssue_1_numeric_c', 'KnowIssue_1_numeric_c',
            'AttitudeExtr2_combined_c', 'Edu_binary', 'Werk_binair', 'Etniciteit_binair'
        ],
    }

    def fmt_cell(model, predictor, italic=False):
        if predictor in model.params.index:
            coef = float(model.params[predictor])
            se = float(model.bse[predictor])
            p = float(model.pvalues[predictor])
            stars = sig_star(p)
            cell = f"{coef:.2f} ({se:.2f}){stars}"
            if p < 0.05:
                cell = f"\\textbf{{{cell}}}"
            if italic:
                cell = f"\\textit{{{cell}}}"
            return cell
        return "--"

    rows = []
    for section_title, keys in sections.items():
        rows.append([f"\\\\[-0.5ex]\n\\multicolumn{{{len(models)+1}}}{{l}}{{\\textbf{{{section_title}}}}} \\\\"])
        for key in keys:
            label = rename_map.get(key, key)
            italic = (section_title == "Control Variables")
            if italic:
                label = f"\\textit{{{label}}}"
            rows.append([label] + [fmt_cell(m, key, italic=italic) for m in models])

    def _n_groups(m, col):
        try:
            fr = m.model.data.frame
            return int(fr[col].nunique())
        except Exception:
            # fall back to fit metadata (post-listwise deletion) if present
            meta = getattr(m, "_fit_meta", {})
            if col == "uid":
                return meta.get("n_coders", np.nan)
            if col == "unit_id":
                return meta.get("n_tweets", np.nan)
            return np.nan

    rows.append(["\\hline"])
    rows.extend([
        ["Var(coder)"]     + [f"{_var_coder(m):.4f}" if np.isfinite(_var_coder(m)) else "NA" for m in models],
        ["Var(tweet)"]     + [f"{_var_tweet(m):.4f}" if np.isfinite(_var_tweet(m)) else "NA" for m in models],
        ["Residual"]       + [f"{_resid_var(m):.4f}" if np.isfinite(_resid_var(m)) else "NA" for m in models],
        ["ICC (coder)"]    + [f"{_icc_coder(m):.4f}" if np.isfinite(_icc_coder(m)) else "NA" for m in models],
        ["ICC (tweet)"]    + [f"{_icc_tweet(m):.4f}" if np.isfinite(_icc_tweet(m)) else "NA" for m in models],
        ["Log Likelihood"] + [f"{m.llf:.2f}" for m in models],
        ["N (Obs)"]        + [f"{int(m.nobs)}" for m in models],
        ["N (Coders)"]     + [f"{_n_groups(m, 'uid')}" for m in models],
        ["N (Tweets)"]     + [f"{_n_groups(m, 'unit_id')}" for m in models],
        ["Fit attempt"]    + [str(getattr(m, "_fit_meta", {}).get("attempt", "NA")) for m in models],
        ["Fit method"]     + [str(getattr(m, "_fit_meta", {}).get("method", "NA")) for m in models],
        ["Fit REML"]       + [str(getattr(m, "_fit_meta", {}).get("reml", "NA")) for m in models],
    ])

    escaped_var = variable_name.replace('_', '\\_')
    header = " & ".join(["Predictor"] + [f"Model {i+1}" for i in range(len(models))]) + " \\\\"
    body = "\n".join(r[0] if r[0].startswith("\\\\") else " & ".join(r) + " \\\\" for r in rows)

    latex = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        f"\\caption{{Stepwise Cross-Classified Mixed-Effects Results for {escaped_var}}}\n"
        f"\\label{{tab:{variable_name}_xclass}}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        f"\\begin{{tabular}}{{{'l' + 'c'*len(models)}}}\n"
        "\\hline\n"
        f"{header}\n"
        "\\hline\n"
        f"{body}\n"
        "\\hline\n"
        "\\end{tabular}%\n"
        "}\n"
        "\\vspace{2mm}\n"
        "\\begin{minipage}{0.95\\textwidth}\\footnotesize\n"
        "\\emph{Note.} Coefficients with standard errors in parentheses. "
        "Bold indicates $p<.05$. Significance stars: $^{*}p<.05$, $^{**}p<.01$, $^{***}p<.001$. "
        "Models include random intercepts for \\emph{coder} (uid) and a \\emph{tweet} variance component. "
        "ICCs are variance components divided by total variance (coder or tweet over coder+tweet+residual). "
        "Left--right self-identification (0--10) and other continuous covariates used in interactions are mean-centered. "
        "Fit diagnostics (attempt/method/REML) are reported for transparency.\n"
        "\\end{minipage}\n"
        "\\end{table}"
    )
    return latex

def tidy_models(models, rename_map):
    out = []
    for mi, m in enumerate(models, start=1):
        for key in m.params.index:
            if key == "Intercept":
                continue
            out.append({
                "model": f"Model {mi}",
                "predictor": key,
                "pretty_name": rename_map.get(key, key),
                "coef": float(m.params[key]),
                "se": float(m.bse[key]),
                "p": float(m.pvalues[key]),
                "stars": sig_star(float(m.pvalues[key])),
                "fit_attempt": getattr(m, "_fit_meta", {}).get("attempt", np.nan),
                "fit_method": getattr(m, "_fit_meta", {}).get("method", ""),
                "fit_reml": getattr(m, "_fit_meta", {}).get("reml", np.nan),
            })
    return pd.DataFrame(out)

# -----------------------------
# 6) Run and upload
# -----------------------------
variables = [
    'stellingen.misinformation',
    'stellingen.toxic',
    'stellingen.sentiment'
]

output_dir_rel = "ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables"
webdav_mkdirs(output_dir_rel)

rename_map = rename_predictors()
fitted_models = {}

for var in variables:
    print(f"\n--- Fitting CROSS-CLASSIFIED models for: {var} ---")
    models = fit_stepwise_models_xclass(df, var, debug=False)
    fitted_models[var] = models

    latex_table = generate_latex_table_xclass(models, rename_map, var)

    tex_rel_path = f"{output_dir_rel}/{var.replace('.', '_')}_models_xclass.tex"
    webdav_upload_bytes(tex_rel_path, latex_table.encode("utf-8"), content_type="text/plain")
    print(f"✅ Uploaded LaTeX: {tex_rel_path}")

    tidy = tidy_models(models, rename_map)
    csv_rel_path = f"{output_dir_rel}/{var.replace('.', '_')}_models_xclass_tidy.csv"
    webdav_upload_bytes(csv_rel_path, tidy.to_csv(index=False).encode("utf-8"), content_type="text/csv")
    print(f"✅ Uploaded Tidy CSV: {csv_rel_path}")

print("\nDone. PolOrient_1_c now appears under CONTROL VARIABLES in the LaTeX tables, "
      "Model 4 includes Congruence × Bias, and variance/ICC extraction is robust.")



--- Fitting CROSS-CLASSIFIED models for: stellingen.misinformation ---
✅ Uploaded LaTeX: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_misinformation_models_xclass.tex
✅ Uploaded Tidy CSV: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_misinformation_models_xclass_tidy.csv

--- Fitting CROSS-CLASSIFIED models for: stellingen.toxic ---
✅ Uploaded LaTeX: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_toxic_models_xclass.tex
✅ Uploaded Tidy CSV: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_toxic_models_xclass_tidy.csv

--- Fitting CROSS-CLASSIFIED models for: stellingen.sentiment ---
✅ Uploaded LaTeX: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_sentiment_models_xclass.tex
✅ Uploaded Tidy CSV: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/stellingen_sentiment_models_xclass_tidy.csv

Done. PolOrient_1_c now appears under CONTROL VARIABLES in the LaTeX tables, Model 4 includes Congruence 

In [2]:
# ============================================================
# Appendix Figure: Ideological congruence effects (publication labels)
# - Panel A: Predicted outcome by congruence × source visibility
# - Panel B: Simple slopes of congruence by instruction condition
#
# Cross-classified MixedLM:
#   - Random intercept for coder (uid)
#   - Tweet variance component (unit_id)
#
# Outputs:
#   - PNG saved locally to ./figures_out
#   - PNG uploaded to Research Drive
# ============================================================

import io
import os
import requests
import numpy as np
import pandas as pd
import patsy
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from rd_utils import webdav_mkdirs, webdav_upload_bytes
import config  # expects BASE_URL, USER, APP_PASSWORD


# -----------------------------
# 0) Settings
# -----------------------------
DEBUG = False

VARS = [
    "stellingen.sentiment",
    "stellingen.misinformation",
    "stellingen.toxic",
]

PRETTY_VAR = {
    "stellingen.sentiment": "Negative Sentiment",
    "stellingen.misinformation": "Misinformation",
    "stellingen.toxic": "Toxicity",
}

SOURCE_LABEL = {
    0: "Source metadata masked",
    1: "Source metadata shown",
}

INSTR_ORDER = ["no instructions", "general instructions", "tailored instructions"]
INSTR_LABEL = {
    "no instructions": "No instructions",
    "general instructions": "General instructions",
    "tailored instructions": "Tailored instructions",
}

# Local output folder (safe anywhere)
LOCAL_OUTDIR = os.path.join(os.getcwd(), "figures_out")
os.makedirs(LOCAL_OUTDIR, exist_ok=True)

# Where to upload
OUTPUT_DIR_REL = "ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/figures"
webdav_mkdirs(OUTPUT_DIR_REL)


# -----------------------------
# 1) Load data
# -----------------------------
BASE = f"{config.BASE_URL}/{config.USER}"
DATA_PATH = ("ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/"
             "final_merged_dataset_for_analysis.parquet")
url = f"{BASE}/{DATA_PATH}"

resp = requests.get(url, auth=(config.USER, config.APP_PASSWORD), stream=True)
resp.raise_for_status()
df = pd.read_parquet(io.BytesIO(resp.content))


# -----------------------------
# 2) Prep: factor ordering + centering
# -----------------------------
df["instruction_type"] = pd.Categorical(
    df["instruction_type"],
    categories=INSTR_ORDER,
    ordered=True
)

if "PolOrient_1" not in df.columns:
    raise KeyError("PolOrient_1 (0–10) not found in dataset.")
df["PolOrient_1"] = pd.to_numeric(df["PolOrient_1"], errors="coerce")


def center_if_present(d: pd.DataFrame, col: str) -> pd.DataFrame:
    if col in d.columns:
        d[col + "_c"] = d[col] - d[col].mean(skipna=True)
    return d


for col in [
    "Age", "PolOrient_1", "vote_likelihood_score", "dsc",
    "ImportIssue_1_numeric", "KnowIssue_1_numeric", "AttitudeExtr2_combined"
]:
    df = center_if_present(df, col)


# -----------------------------
# 3) Fit helpers
# -----------------------------
def _patsy_listwise_drop(formula: str, data: pd.DataFrame) -> pd.DataFrame:
    _, X = patsy.dmatrices(formula, data, return_type="dataframe", NA_action="drop")
    return data.loc[X.index].copy()


def fit_xclass(formula: str, data: pd.DataFrame,
               group_col: str = "uid", tweet_col: str = "unit_id"):
    d = _patsy_listwise_drop(formula, data)

    attempts = [
        {"reml": True,  "method": "lbfgs", "kwargs": {}},
        {"reml": False, "method": "lbfgs", "kwargs": {}},
        {"reml": False, "method": "nm",    "kwargs": {"maxiter": 2000, "disp": False}},
    ]

    last_err = None
    for a in attempts:
        try:
            if DEBUG:
                print(f"[FIT] reml={a['reml']} method={a['method']} n={len(d)}")
            res = smf.mixedlm(
                formula, data=d,
                groups=d[group_col],
                re_formula="1",
                vc_formula={"tweet": f"0 + C({tweet_col})"}
            ).fit(reml=a["reml"], method=a["method"], **a["kwargs"])
            res._fit_frame = d  # post-listwise data used
            return res
        except Exception as e:
            last_err = e
            if DEBUG:
                print(f"[FIT] failed: {type(e).__name__}: {e}")

    raise RuntimeError(f"Model failed for formula:\n{formula}\nLast error: {last_err}")


def fixed_effect_ci_band(m, newdata: pd.DataFrame):
    """
    95% CI for the fixed-effect linear predictor at each row of newdata.
    Uses patsy + fixed-effect covariance.
    """
    rhs = m.model.formula.split("~", 1)[1].strip()
    X_new = patsy.dmatrix(rhs, newdata, return_type="dataframe")

    fe_names = list(m.fe_params.index)
    X = X_new.loc[:, fe_names].to_numpy()

    cov = m.cov_params()
    cov_fe = cov.loc[fe_names, fe_names].to_numpy()

    pred = X @ m.fe_params.to_numpy()
    se = np.sqrt(np.maximum(0.0, np.einsum("ij,jk,ik->i", X, cov_fe, X)))

    z = 1.96
    lo = pred - z * se
    hi = pred + z * se
    return pred, lo, hi


def lincomb_ci(m, L: np.ndarray):
    fe_names = list(m.fe_params.index)
    cov = m.cov_params().loc[fe_names, fe_names].to_numpy()
    beta = m.fe_params.to_numpy()

    est = float(L @ beta)
    se = float(np.sqrt(max(0.0, L @ cov @ L)))
    lo = est - 1.96 * se
    hi = est + 1.96 * se
    return est, lo, hi


# -----------------------------
# 4) Formulas (match your hypotheses)
# -----------------------------
FORMULA_A = (
    "value_scaled ~ Male + Age_c + PolOrient_1_c + instruction_type + dsc_c + "
    "source_shown * vote_likelihood_score_c + "
    "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
    "Edu_binary + Werk_binair + Etniciteit_binair"
)

FORMULA_B = (
    "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + dsc_c + "
    "vote_likelihood_score_c * instruction_type + "
    "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
    "Edu_binary + Werk_binair + Etniciteit_binair"
)


# -----------------------------
# 5) Plot builder
# -----------------------------
def make_figure_for_var(dfull: pd.DataFrame, varname: str):
    d = dfull[dfull["variable"] == varname].copy()

    mA = fit_xclass(FORMULA_A, d)
    mB = fit_xclass(FORMULA_B, d)

    # numeric centered controls at 0; binaries at sample mean
    base = {
        "Age_c": 0.0,
        "PolOrient_1_c": 0.0,
        "dsc_c": 0.0,
        "ImportIssue_1_numeric_c": 0.0,
        "KnowIssue_1_numeric_c": 0.0,
        "AttitudeExtr2_combined_c": 0.0,
        "vote_likelihood_score_c": 0.0,  # overwritten per grid
    }
    for b in ["Male", "Edu_binary", "Werk_binair", "Etniciteit_binair", "source_shown"]:
        base[b] = float(pd.to_numeric(d[b], errors="coerce").mean(skipna=True)) if b in d.columns else 0.0

    x_grid = np.linspace(-1.0, 3.0, 60)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=200)
    fig.suptitle(PRETTY_VAR.get(varname, varname), y=1.02)

    # Panel A
    for src in [0, 1]:
        nd = pd.DataFrame({
            **base,
            "source_shown": src,
            "vote_likelihood_score_c": x_grid,
            "instruction_type": "no instructions",
        })
        nd["instruction_type"] = pd.Categorical(nd["instruction_type"], categories=INSTR_ORDER, ordered=True)

        pred, lo, hi = fixed_effect_ci_band(mA, nd)
        ax1.plot(x_grid, pred, label=SOURCE_LABEL[src])
        ax1.fill_between(x_grid, lo, hi, alpha=0.18)

    ax1.set_title("A. Congruence × source visibility")
    ax1.set_xlabel("Ideological congruence (centered)")
    ax1.set_ylabel("Predicted rating")
    ax1.legend(frameon=False)

    # Panel B (simple slopes)
    fe_names = list(mB.fe_params.index)
    t_cong = "vote_likelihood_score_c"
    t_gen = "vote_likelihood_score_c:instruction_type[T.general instructions]"
    t_tai = "vote_likelihood_score_c:instruction_type[T.tailored instructions]"

    if t_cong not in fe_names:
        raise RuntimeError("vote_likelihood_score_c not in fixed effects; check FORMULA_B.")

    def L_for(terms):
        L = np.zeros(len(fe_names), dtype=float)
        for t in terms:
            if t not in fe_names:
                raise KeyError(f"Expected term '{t}' not in model fixed effects.\nGot: {fe_names}")
            L[fe_names.index(t)] += 1.0
        return L

    est0, lo0, hi0 = lincomb_ci(mB, L_for([t_cong]))
    est1, lo1, hi1 = lincomb_ci(mB, L_for([t_cong, t_gen]))
    est2, lo2, hi2 = lincomb_ci(mB, L_for([t_cong, t_tai]))

    labels = [INSTR_LABEL["no instructions"], INSTR_LABEL["general instructions"], INSTR_LABEL["tailored instructions"]]
    y = np.array([est0, est1, est2])
    cis = [(lo0, hi0), (lo1, hi1), (lo2, hi2)]
    yerr = np.vstack([y - np.array([c[0] for c in cis]), np.array([c[1] for c in cis]) - y])

    x = np.arange(len(labels))
    ax2.axhline(0.0, linewidth=1.2, alpha=0.7)
    ax2.errorbar(x, y, yerr=yerr, fmt="o", capsize=6)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.set_title("B. Congruence slopes by instruction")
    ax2.set_ylabel("Congruence slope (β)")

    fig.tight_layout()
    return fig


# -----------------------------
# 6) Generate + upload
# -----------------------------
for var in VARS:
    print(f"--- Plotting appendix figure for: {var} ---")
    fig = make_figure_for_var(df, var)

    fname = f"{var.replace('.', '_')}_appendix_congruence_panels.png"
    local_path = os.path.join(LOCAL_OUTDIR, fname)

    fig.savefig(local_path, bbox_inches="tight")
    plt.close(fig)

    with open(local_path, "rb") as f:
        img_bytes = f.read()

    remote_path = f"{OUTPUT_DIR_REL}/{fname}"
    webdav_upload_bytes(remote_path, img_bytes, content_type="image/png")
    print(f"✅ Saved locally: {local_path}")
    print(f"✅ Uploaded: {remote_path}")

print("Done.")


--- Plotting appendix figure for: stellingen.sentiment ---
✅ Saved locally: /home/akroon/AnnotationBias/figures_out/stellingen_sentiment_appendix_congruence_panels.png
✅ Uploaded: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/figures/stellingen_sentiment_appendix_congruence_panels.png
--- Plotting appendix figure for: stellingen.misinformation ---
✅ Saved locally: /home/akroon/AnnotationBias/figures_out/stellingen_misinformation_appendix_congruence_panels.png
✅ Uploaded: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/figures/stellingen_misinformation_appendix_congruence_panels.png
--- Plotting appendix figure for: stellingen.toxic ---
✅ Saved locally: /home/akroon/AnnotationBias/figures_out/stellingen_toxic_appendix_congruence_panels.png
✅ Uploaded: ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/figures/stellingen_toxic_appendix_congruence_panels.png
Done.


In [ ]:
import numpy as np
import pandas as pd
import patsy
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D

# -----------------------------
# 0) Settings
# -----------------------------
DVS = ["stellingen.misinformation", "stellingen.toxic", "stellingen.sentiment"]
DV_TITLES = {
    "stellingen.misinformation": "Misinformation",
    "stellingen.toxic": "Toxicity",
    "stellingen.sentiment": "Negative Sentiment",
}

# Prefer dsc (raw) for splitting; fall back to centered dsc_c
DSC_SPLIT_COL = "dsc" if "dsc" in df.columns else "dsc_c"

GROUPS = ["Low bias", "High bias", "Pooled"]
offset_map = {"Low bias": +0.22, "High bias": 0.00, "Pooled": -0.22}

group_colors = {
    "Low bias": "tab:blue",
    "High bias": "tab:orange",
    "Pooled": "tab:gray",
}

MODEL_FORMULA = (
    "value_scaled ~ Male + Age_c + PolOrient_1_c + source_shown + vote_likelihood_score_c + "
    "instruction_type + dsc_c + vote_likelihood_score_c:dsc_c + "
    "ImportIssue_1_numeric_c + KnowIssue_1_numeric_c + AttitudeExtr2_combined_c + "
    "Edu_binary + Werk_binair + Etniciteit_binair"
)

PLOT_ORDER = [
    "instruction_type[T.general instructions]",
    "instruction_type[T.tailored instructions]",
    "source_shown",
    "dsc_c",
    "vote_likelihood_score_c",
    "vote_likelihood_score_c:dsc_c",
    "Male",
    "Age_c",
    "PolOrient_1_c",
    "ImportIssue_1_numeric_c",
    "KnowIssue_1_numeric_c",
    "AttitudeExtr2_combined_c",
    "Edu_binary",
    "Werk_binair",
    "Etniciteit_binair",
]

PLOT_LABELS = {
    "instruction_type[T.general instructions]": "General instructions",
    "instruction_type[T.tailored instructions]": "Tailored instructions",
    "source_shown": "Source metadata shown",
    "dsc_c": "Implicit bias (centered)",
    "vote_likelihood_score_c": "Ideological congruence (centered)",
    "vote_likelihood_score_c:dsc_c": "Congruence × Bias",
    "Male": "Male",
    "Age_c": "Age (centered)",
    "PolOrient_1_c": "Left–right self-ID (centered)",
    "ImportIssue_1_numeric_c": "Issue importance (centered)",
    "KnowIssue_1_numeric_c": "Issue knowledge (centered)",
    "AttitudeExtr2_combined_c": "Anti-immigrant attitude (centered)",
    "Edu_binary": "Higher educated",
    "Werk_binair": "Employed",
    "Etniciteit_binair": "Dutch ethnicity",
}

# For display only: suppress extreme garbage CIs
MAX_CI_WIDTH = 5.0

# -----------------------------
# 1) Clean + make 2-group split
# -----------------------------
df = df.copy()
df.columns = df.columns.str.strip()
df["variable"] = df["variable"].astype(str).str.strip()

df["instruction_type"] = df["instruction_type"].astype(str).str.strip()
df["instruction_type"] = pd.Categorical(
    df["instruction_type"],
    categories=["no instructions", "general instructions", "tailored instructions"],
    ordered=True
)

df["value_scaled"] = pd.to_numeric(df["value_scaled"], errors="coerce")
df[DSC_SPLIT_COL] = pd.to_numeric(df[DSC_SPLIT_COL], errors="coerce")

# Median split on implicit bias
med = df[DSC_SPLIT_COL].median(skipna=True)
df["dsc_group2"] = np.where(df[DSC_SPLIT_COL] <= med, "Low bias", "High bias")

# -----------------------------
# 2) Helpers
# -----------------------------
def listwise_drop_patsy(formula, data):
    y, X = patsy.dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    return data.loc[X.index].copy()

def fit_xclass(formula, data, group_col="uid", tweet_col="unit_id"):
    d = listwise_drop_patsy(formula, data)

    if len(d) < 25 or d[group_col].nunique() < 2 or d[tweet_col].nunique() < 2:
        return None, d

    try:
        m = smf.mixedlm(
            formula, data=d,
            groups=d[group_col],
            re_formula="1",
            vc_formula={"tweet": f"0 + C({tweet_col})"}
        ).fit(reml=True, method="lbfgs")
    except Exception:
        try:
            m = smf.mixedlm(
                formula, data=d,
                groups=d[group_col],
                re_formula="1",
                vc_formula={"tweet": f"0 + C({tweet_col})"}
            ).fit(reml=False, method="lbfgs")
        except Exception:
            m = smf.mixedlm(
                formula, data=d,
                groups=d[group_col],
                re_formula="1",
                vc_formula={"tweet": f"0 + C({tweet_col})"}
            ).fit(reml=False, method="nm", maxiter=2000, disp=False)

    return m, d

def extract_params(m, dv, group):
    rows = []
    if m is None:
        return rows

    for key in PLOT_ORDER:
        if key not in m.params.index:
            continue
        est = float(m.params[key])
        se = float(m.bse[key])
        p = float(m.pvalues[key])
        rows.append({
            "dv": dv,
            "group": group,
            "predictor_clean": PLOT_LABELS.get(key, key),
            "estimate": est,
            "ci_lower": est - 1.96 * se,
            "ci_upper": est + 1.96 * se,
            "significant": p < 0.05,
        })
    return rows

# -----------------------------
# 3) Fit all (Low, High, Pooled)
# -----------------------------
plot_rows = []
diag_rows = []

for dv in DVS:
    for group in GROUPS:
        if group == "Pooled":
            subset = df[df["variable"] == dv].copy()
        else:
            subset = df[(df["variable"] == dv) & (df["dsc_group2"] == group)].copy()

        m, d_used = fit_xclass(MODEL_FORMULA, subset)

        diag_rows.append({
            "dv": dv, "group": group,
            "n_before": len(subset),
            "n_after": len(d_used),
            "uid_n": d_used["uid"].nunique() if len(d_used) else 0,
            "tweet_n": d_used["unit_id"].nunique() if len(d_used) else 0,
            "fitted": m is not None,
        })

        plot_rows.extend(extract_params(m, dv, group))

diag_df = pd.DataFrame(diag_rows)
print("\n=== Diagnostics ===")
print(diag_df.sort_values(["dv", "group"]).to_string(index=False))

plot_df = pd.DataFrame(plot_rows)
if plot_df.empty:
    raise RuntimeError("No model results extracted.")

# suppress ridiculous CIs for display
plot_df["ci_width"] = plot_df["ci_upper"] - plot_df["ci_lower"]
plot_df["exploded"] = plot_df["ci_width"] > MAX_CI_WIDTH

# -----------------------------
# 4) y positions
# -----------------------------
ordered_labels = [PLOT_LABELS[k] for k in PLOT_ORDER]
K = len(ordered_labels)
y_base = {lab: i for i, lab in enumerate(ordered_labels)}

plot_df["y_base"] = plot_df["predictor_clean"].map(y_base)
plot_df["y"] = plot_df["y_base"] + plot_df["group"].map(offset_map)

# -----------------------------
# 5) Plot
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 12), sharey=True)

for i, dv in enumerate(DVS):
    ax = axes[i]
    dsub = plot_df[plot_df["dv"] == dv].copy()

    # robust limits from non-exploded rows
    normal = dsub[~dsub["exploded"]]
    bounds = np.r_[normal["ci_lower"].to_numpy(), normal["ci_upper"].to_numpy()]
    bounds = bounds[np.isfinite(bounds)]
    lim = np.quantile(np.abs(bounds), 0.98) if bounds.size else 1.0
    lim = max(lim, 0.25)
    xlim = (-lim, lim)

    for group in GROUPS:
        g = dsub[dsub["group"] == group].copy()
        g = g[~g["exploded"]]  # display-only
        if g.empty:
            continue

        col = group_colors[group]

        for _, row in g.iterrows():
            alpha = 1.0 if row["significant"] else 0.25
            lo = np.clip(row["ci_lower"], xlim[0], xlim[1])
            hi = np.clip(row["ci_upper"], xlim[0], xlim[1])
            est = np.clip(row["estimate"], xlim[0], xlim[1])

            ax.errorbar(
                est, row["y"],
                xerr=[[est - lo], [hi - est]],
                fmt="o",
                color=col, ecolor=col,
                alpha=alpha,
                capsize=3,
                markersize=5,
                elinewidth=1
            )

    ax.axvline(x=0, linestyle="dotted", linewidth=1)
    ax.set_title(DV_TITLES.get(dv, dv), fontsize=14, weight="bold")
    ax.set_xlabel("Coefficient")
    ax.grid(True, axis="x", linestyle="--", alpha=0.5)
    ax.set_xlim(xlim)

    ax.set_yticks(range(K))
    ax.set_yticklabels(ordered_labels)
    ax.set_ylim(K - 0.5, -0.5)

handles = [Line2D([0], [0], marker="o", linestyle="None", color=group_colors[g], label=g) for g in GROUPS]
fig.legend(
    handles=handles,
    labels=[h.get_label() for h in handles],
    title="Implicit bias group (median split)\n(alpha = non-significant)",
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.05)
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
